# 01 — HSV Team Colour Calibration

Interactive notebook for identifying HSV colour ranges used to classify players by team.


In [ ]:
%pip install ipywidgets==8.1 jupyterlab_widgets

In [ ]:
#Correr se código abaixo não der
#import sys
#!{sys.executable} -m pip install -U "ipywidgets>=8,<9" "jupyterlab_widgets>=3,<4"

In [ ]:
# === HSV Picker robusto para JupyterLab (upload + path, v7/v8) ===
import os, cv2, numpy as np, matplotlib.pyplot as plt
import ipywidgets as W
from IPython.display import display

# ---------------- Widgets ----------------
uploader = W.FileUpload(accept="image/*", multiple=False)
path_txt = W.Text(value="", description="Path:", placeholder=r"C:\caminho\imagem.png", layout=W.Layout(width="60%"))
load_btn = W.Button(description="Carregar do disco")
render_btn = W.Button(description="Renderizar", button_style="primary")
status = W.HTML(value="⬆️ Carrega uma imagem (upload) ou indica um Path e clica Carregar.")

# Sliders (atualização contínua)
h1 = W.IntRangeSlider(value=[0,179], min=0, max=179, step=1, description='H1', continuous_update=True, layout=W.Layout(width="60%"))
use_h2 = W.Checkbox(value=False, description='Usar H2 (wrap)')
h2 = W.IntRangeSlider(value=[170,179], min=0, max=179, step=1, description='H2', continuous_update=True, disabled=True, layout=W.Layout(width="60%"))
s  = W.IntRangeSlider(value=[0,255], min=0, max=255, step=1, description='S',  continuous_update=True, layout=W.Layout(width="60%"))
v  = W.IntRangeSlider(value=[0,255], min=0, max=255, step=1, description='V',  continuous_update=True, layout=W.Layout(width="60%"))

ranges_box = W.Textarea(value="", description="Ranges", layout=W.Layout(width="100%", height="70px"))
out = W.Output()

# ---------------- Estado ----------------
img0 = None           # imagem original (BGR)
max_display = 900     # redimensiona para fluidez

# ---------------- Utilitários ----------------
def decode_upload(val):
    """
    Lê bytes do FileUpload em ipywidgets v7 (tuple/list de dict/Bunch) ou v8 (dict).
    Devolve np.ndarray BGR ou None.
    """
    if not val:
        return None
    try:
        # v8: dict {filename: {content: bytes, ...}}
        if isinstance(val, dict):
            first_key = next(iter(val))
            content = val[first_key].get("content", None)
        else:
            # v7: tuple/list com dicts/Bunch
            first = val[0]
            if isinstance(first, dict):
                content = first.get("content")
            else:
                # Bunch-like
                content = getattr(first, "content", None)
        if content is None:
            raise ValueError("Campo 'content' não encontrado no upload.")
        arr = np.frombuffer(content, dtype=np.uint8)
        img = cv2.imdecode(arr, cv2.IMREAD_COLOR)
        return img
    except Exception as e:
        status.value = f"<b>Erro ao ler upload:</b> {e}"
        return None

def load_from_path(p):
    if not p or not os.path.exists(p):
        status.value = f"<b>⚠️ Path inválido</b>: {p}"
        return None
    img = cv2.imread(p)
    if img is None:
        status.value = f"<b>⚠️ Não consegui abrir</b>: {p}"
    return img

def make_mask(hsv, h1_rng, s_rng, v_rng, h2_rng=None):
    lo1 = np.array([h1_rng[0], s_rng[0], v_rng[0]], np.uint8)
    hi1 = np.array([h1_rng[1], s_rng[1], v_rng[1]], np.uint8)
    m = cv2.inRange(hsv, lo1, hi1)
    if h2_rng is not None:
        lo2 = np.array([h2_rng[0], s_rng[0], v_rng[0]], np.uint8)
        hi2 = np.array([h2_rng[1], s_rng[1], v_rng[1]], np.uint8)
        m |= cv2.inRange(hsv, lo2, hi2)
    return m

def spec_text():
    spec = {"h1": tuple(h1.value), "s": tuple(s.value), "v": tuple(v.value)}
    if use_h2.value:
        spec["h2"] = tuple(h2.value)
    return "TEAM_HSV = " + repr(spec)

# ---------------- Renderização ----------------
def render(_=None):
    if img0 is None:
        status.value = "<b>⚠️ Falta imagem.</b> Usa upload ou path."
        return

    hsv = cv2.cvtColor(img0, cv2.COLOR_BGR2HSV)
    h2_rng = tuple(h2.value) if use_h2.value else None
    mask = make_mask(hsv, tuple(h1.value), tuple(s.value), tuple(v.value), h2_rng)
    cov = float((mask > 0).mean() * 100.0)

    disp = img0
    H, W = disp.shape[:2]
    scale = min(1.0, max_display / max(H, W))
    if scale < 1.0:
        disp = cv2.resize(disp, (int(W*scale), int(H*scale)), interpolation=cv2.INTER_AREA)
        mask_disp = cv2.resize(mask, (disp.shape[1], disp.shape[0]), interpolation=cv2.INTER_NEAREST)
    else:
        mask_disp = mask

    img_rgb = cv2.cvtColor(disp, cv2.COLOR_BGR2RGB)
    masked  = cv2.bitwise_and(disp, disp, mask=mask_disp)
    masked  = cv2.cvtColor(masked, cv2.COLOR_BGR2RGB)

    ranges_box.value = spec_text()
    status.value = f"<b>OK.</b> Cobertura: <b>{cov:.1f}%</b>. Ajusta H/S/V."

    with out:
        out.clear_output(wait=True)
        fig, ax = plt.subplots(1, 2, figsize=(10, 5))
        ax[0].imshow(img_rgb); ax[0].set_title("Original"); ax[0].axis("off")
        ax[1].imshow(masked);  ax[1].set_title(f"Mask ▶ ({cov:.1f}%)"); ax[1].axis("off")
        plt.show()

# ---------------- Callbacks ----------------
def on_upload(change):
    global img0
    img0 = decode_upload(uploader.value)
    render()

def on_load_btn(_):
    global img0
    img0 = load_from_path(path_txt.value)
    render()

def on_toggle_h2(change):
    h2.disabled = not change["new"]
    render()

# ligações
uploader.observe(on_upload, names="value")
load_btn.on_click(on_load_btn)
render_btn.on_click(render)
use_h2.observe(on_toggle_h2, names="value")
for w in (h1, h2, s, v):
    w.observe(render, names="value")

# ---------------- UI ----------------
ui = W.VBox([
    W.HBox([uploader]),
    W.HBox([path_txt, load_btn, render_btn]),
    W.HBox([h1]),
    W.HBox([use_h2, h2]),
    W.HBox([s]),
    W.HBox([v]),
    ranges_box,
    status,
    out
])

display(ui)
